In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 110

RAW = Path(r'C:\Users\Rono\Desktop\GEE\Data\Processed')
OUT = Path(r'C:\Users\Rono\Desktop\GEE\Data\Processed')
FIG = Path(r"C:\Users\Rono\Desktop\GEE\Figures")
RES = Path(r'C:\Users\Rono\Desktop\GEE\Results')

for p in [OUT, FIG, RES]:
    p.mkdir(parents=True, exist_ok=True)

# --- Column map: declares what each raw column means ---
# GEE names columns based on input bands:
#   single-band input -> 'mean', 'stdDev', 'count'
#   multi-band input  -> '<band>_mean', '<band>_stdDev', '<band>_count'
COLUMN_MAP = {
    's2': {
        'mean':   'ndvi_s2',
        'stdDev': 'ndvi_s2_std',
        'count':  'ndvi_s2_count',
        'scale':  0.0001,   # raw stored as Int16 * 10000
    },
    'modis': {
        'NDVI_mean':   'ndvi_modis',
        'NDVI_stdDev': 'ndvi_modis_std',
        'NDVI_count':  'ndvi_modis_count',
        'EVI_mean':    'evi_modis',
        'EVI_stdDev':  'evi_modis_std',
        'EVI_count':   'evi_modis_count',
        'scale':       1.0,   # already scaled in GEE
    },
    'chirps': {
        'mean':   'precip_mm',
        'stdDev': 'precip_std',
        'count':  'precip_count',
        'scale':  1.0,
    },
    'era5': {
        'temp_2m_c':      'temp_2m_c',
        'precip_mm':      'precip_era5_mm',
        'evaporation_mm': 'evaporation_mm',
        'sm_layer1':      'sm_era5_l1',
        'sm_layer2':      'sm_era5_l2',
        'sm_layer3':      'sm_era5_l3',
        'sm_layer4':      'sm_era5_l4',
        'scale':          1.0,
    },
    'fapar': {
        'LAI_mean':     'lai',
        'LAI_stdDev':   'lai_std',
        'FAPAR_mean':   'fapar',
        'FAPAR_stdDev': 'fapar_std',
        'scale':        1.0,
    },
    'smap': {
        'sm_surface_mean':    'sm_surface',
        'sm_surface_stdDev':  'sm_surface_std',
        'sm_surface_count':   'sm_surface_count',
        'sm_rootzone_mean':   'sm_rootzone',
        'sm_rootzone_stdDev': 'sm_rootzone_std',
        'sm_rootzone_count':  'sm_rootzone_count',
        'scale':              1.0,
    },
    'lc': {
        'mode': 'lc_class',
    },
}

FILES = {
    's2':     'Kericho_Wards_NDVI_S2_2020_2025.csv',
    'modis':  'Kericho_Wards_MODIS_2000_2025.csv',
    'chirps': 'Kericho_Wards_CHIRPS_2020_2025.csv',
    'era5':   'Kericho_Wards_ERA5_2020_2025.csv',
    'fapar':  'Kericho_Wards_FAPAR_2020_2025.csv',
    'smap':   'Kericho_Wards_SMAP_2020_2025.csv',
    'lc':     'Kericho_Wards_LandCover_2019.csv',
}

def standardize(df, key):
    spec = COLUMN_MAP[key]
    rename = {k: v for k, v in spec.items()
              if k != 'scale' and k in df.columns}
    df = df.rename(columns=rename)

    scale = spec.get('scale', 1.0)
    if scale != 1.0:
        for col in df.columns:
            if col.startswith(('ndvi_', 'evi_')) and not col.endswith('_count'):
                df[col] = df[col] * scale

    if 'month' in df.columns:
        df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

    if 'ward_name' in df.columns:
        df['ward_name'] = df['ward_name'].astype(str).str.strip()

    return df

In [2]:
raw = {}
for key, fname in FILES.items():
    df = pd.read_csv(RAW / fname)
    raw[key] = df
    print(f"{key:8s} shape={df.shape}  cols={df.shape[1]}")

s2       shape=(2160, 7)  cols=7
modis    shape=(9360, 10)  cols=10
chirps   shape=(2160, 7)  cols=7
era5     shape=(2160, 11)  cols=11
fapar    shape=(2160, 10)  cols=10
smap     shape=(2160, 10)  cols=10
lc       shape=(30, 4)  cols=4


In [3]:
s2     = standardize(raw['s2'],     's2')
modis  = standardize(raw['modis'],  'modis')
chirps = standardize(raw['chirps'], 'chirps')
era5   = standardize(raw['era5'],   'era5')
fapar  = standardize(raw['fapar'],  'fapar')
smap   = standardize(raw['smap'],   'smap')
lc     = standardize(raw['lc'],     'lc')

print('s2 columns:    ', s2.columns.tolist())
print('chirps columns:', chirps.columns.tolist())
print('era5 columns:  ', era5.columns.tolist())
print('smap columns:  ', smap.columns.tolist())

s2 columns:     ['ward_name', 'ward_id', 'county', 'month', 'ndvi_s2', 'ndvi_s2_std', 'ndvi_s2_count']
chirps columns: ['ward_name', 'ward_id', 'county', 'month', 'precip_mm', 'precip_std', 'precip_count']
era5 columns:   ['ward_name', 'ward_id', 'county', 'month', 'temp_2m_c', 'precip_era5_mm', 'evaporation_mm', 'sm_era5_l1', 'sm_era5_l2', 'sm_era5_l3', 'sm_era5_l4']
smap columns:   ['ward_name', 'ward_id', 'county', 'month', 'sm_surface', 'sm_surface_std', 'sm_surface_count', 'sm_rootzone', 'sm_rootzone_std', 'sm_rootzone_count']


In [4]:
monthly = {'s2': s2, 'chirps': chirps, 'era5': era5,
           'fapar': fapar, 'smap': smap}

# 4a. Duplicated (ward, month) keys
for name, df in monthly.items():
    dups = df.duplicated(subset=['ward_name', 'month']).sum()
    print(f"{name:8s} duplicate keys: {dups}")
    assert dups == 0, f"{name} has duplicate (ward, month) rows"

# 4b. Same 30 wards everywhere
ward_sets = {name: set(df['ward_name'].unique()) for name, df in monthly.items()}
reference = ward_sets['s2']
for name, wards in ward_sets.items():
    assert wards == reference, f"{name} wards differ from s2"
print(f"\nAll files share {len(reference)} wards")

# 4c. Month coverage
for name, df in monthly.items():
    print(f"{name:8s} months: {df['month'].min().date()} to {df['month'].max().date()}, "
          f"n={df['month'].nunique()}")

# 4d. No nulls in key or value columns
for name, df in monthly.items():
    nulls = df.isnull().sum()
    bad = nulls[nulls > 0]
    if len(bad):
        print(f"{name} nulls:\n{bad}")
    else:
        print(f"{name:8s} no nulls")

# 4e. Land cover: one row per ward
assert len(lc) == 30, f"Land cover should have 30 rows, has {len(lc)}"
print(f"\nLand cover classes present: {sorted(lc['lc_class'].unique())}")

s2       duplicate keys: 0
chirps   duplicate keys: 0
era5     duplicate keys: 0
fapar    duplicate keys: 0
smap     duplicate keys: 0

All files share 30 wards
s2       months: 2020-01-01 to 2025-12-01, n=72
chirps   months: 2020-01-01 to 2025-12-01, n=72
era5     months: 2020-01-01 to 2025-12-01, n=72
fapar    months: 2020-01-01 to 2025-12-01, n=72
smap     months: 2020-01-01 to 2025-12-01, n=72
s2       no nulls
chirps   no nulls
era5     no nulls
fapar nulls:
lai          190
lai_std      190
fapar        190
fapar_std    190
dtype: int64
smap     no nulls

Land cover classes present: [np.float64(39.99999999999996), np.float64(39.99999999999997), np.float64(39.99999999999998), np.float64(39.99999999999999), np.float64(40.0), np.float64(40.00000000000001), np.float64(40.000000000000014), np.float64(40.00000000000002), np.float64(40.00000000000003), np.float64(40.00000000000005), np.float64(111.9999999999999), np.float64(126.0), np.float64(126.00000000000006)]


In [7]:
# investigation 
# Count zero-NDVI rows (should be the empty-month placeholders)
zeros = modis[modis['ndvi_modis'] == 0]
print(f"Zero-NDVI rows: {len(zeros)}")
print(zeros[['ward_name', 'month', 'ndvi_modis', 'evi_modis']].head(20))

# Which months have zeros?
print("\nMonths with zero-NDVI placeholders:")
print(zeros['month'].dt.to_period('M').value_counts().sort_index())

# Which wards are affected?
print("\nWards with zero-NDVI placeholders:")
print(zeros.groupby('ward_name').size().sort_values(ascending=False))

# Find the 4 missing rows
expected = pd.MultiIndex.from_product(
    [modis['ward_name'].unique(),
     pd.date_range('2000-01-01', '2025-12-01', freq='MS')],
    names=['ward_name', 'month']
)
actual = pd.MultiIndex.from_frame(modis[['ward_name', 'month']])
missing = expected.difference(actual)
print(f"\nMissing rows: {len(missing)}")
print(missing.tolist())

Zero-NDVI rows: 30
               ward_name      month  ndvi_modis  evi_modis
0                Ainamoi 2000-01-01         0.0        0.0
1            Kapkugerwet 2000-01-01         0.0        0.0
2                Kapsaos 2000-01-01         0.0        0.0
3                Kapsoit 2000-01-01         0.0        0.0
4              Kipchebor 2000-01-01         0.0        0.0
5            Kipchimchim 2000-01-01         0.0        0.0
6                  Chaik 2000-01-01         0.0        0.0
7   Cheptororiet/Seretut 2000-01-01         0.0        0.0
8               Kabianga 2000-01-01         0.0        0.0
9               Kapsuser 2000-01-01         0.0        0.0
10                Waldai 2000-01-01         0.0        0.0
11               Cheboin 2000-01-01         0.0        0.0
12              Chemosot 2000-01-01         0.0        0.0
13            Cheplanget 2000-01-01         0.0        0.0
14              Kapkatet 2000-01-01         0.0        0.0
15               Kisiara 2000-01-01  

print('\n--- MODIS NDVI and EVI ---')

# Drop empty-month placeholders from Jan 2000 (pre-Terra-launch gap).
placeholder_mask = (modis['ndvi_modis'] == 0) & (modis['evi_modis'] == 0)
n_placeholders = placeholder_mask.sum()
if n_placeholders:
    print(f"Dropping {n_placeholders} placeholder rows "
          f"({modis.loc[placeholder_mask, 'month'].min().date()})")
    modis = modis[~placeholder_mask].copy()

# Drop rows where MODIS returned null (known Oct 2009 and Oct 2015 gaps).
before = len(modis)
modis = modis.dropna(subset=['ndvi_modis', 'evi_modis']).copy()
n_nan = before - len(modis)
if n_nan:
    print(f"Dropping {n_nan} rows with null MODIS values "
          f"(known Oct 2009 / Oct 2015 sensor gaps)")

print(f"Final MODIS rows: {len(modis)}")
print(modis[['ndvi_modis', 'evi_modis']].describe())

# Assertions with printed offending values on failure.
ndvi_bad = modis.loc[~modis['ndvi_modis'].between(-0.1, 1.05), 'ndvi_modis']
assert len(ndvi_bad) == 0, f"Out-of-range NDVI: {ndvi_bad.unique()[:5]}"

evi_bad = modis.loc[~modis['evi_modis'].between(-0.1, 1.05), 'evi_modis']
assert len(evi_bad) == 0, f"Out-of-range EVI: {evi_bad.unique()[:5]}"

In [12]:
# --- MODIS NDVI and EVI ---
print('--- MODIS NDVI and EVI ---')

# Drop Jan 2000 placeholder rows (pre-Terra-launch gap; both bands zero).
placeholder_mask = (modis['ndvi_modis'] == 0) & (modis['evi_modis'] == 0)
n_placeholders = placeholder_mask.sum()
if n_placeholders:
    print(f"Dropping {n_placeholders} placeholder rows "
          f"({modis.loc[placeholder_mask, 'month'].min().date()})")
    modis = modis[~placeholder_mask].copy()

# Drop rows where MODIS returned null (known Oct 2009 and Oct 2015 gaps).
before = len(modis)
modis = modis.dropna(subset=['ndvi_modis', 'evi_modis']).copy()
n_nan = before - len(modis)
if n_nan:
    print(f"Dropping {n_nan} rows with null MODIS values "
          f"(known Oct 2009 / Oct 2015 sensor gaps)")

print(f"Final MODIS rows: {len(modis)}")
print(modis[['ndvi_modis', 'evi_modis']].describe())

ndvi_bad = modis.loc[~modis['ndvi_modis'].between(-0.1, 1.05), 'ndvi_modis']
assert len(ndvi_bad) == 0, f"Out-of-range MODIS NDVI: {ndvi_bad.unique()[:5]}"

evi_bad = modis.loc[~modis['evi_modis'].between(-0.1, 1.05), 'evi_modis']
assert len(evi_bad) == 0, f"Out-of-range MODIS EVI: {evi_bad.unique()[:5]}"


# --- Sentinel-2 NDVI ---
print('\n--- Sentinel-2 NDVI ---')
s2 = s2.dropna(subset=['ndvi_s2']).copy()
print(s2['ndvi_s2'].describe())

s2_bad = s2.loc[~s2['ndvi_s2'].between(-0.1, 1.05), 'ndvi_s2']
assert len(s2_bad) == 0, f"Out-of-range S2 NDVI: {s2_bad.unique()[:5]}"


# --- CHIRPS precipitation (mm/month) ---
print('\n--- CHIRPS precipitation (mm/month) ---')
chirps = chirps.dropna(subset=['precip_mm']).copy()
print(chirps['precip_mm'].describe())

chirps_bad = chirps.loc[~chirps['precip_mm'].between(0, 800), 'precip_mm']
assert len(chirps_bad) == 0, f"Out-of-range CHIRPS rainfall: {chirps_bad.unique()[:5]}"


# --- ERA5 temperature (°C) ---
print('\n--- ERA5 temperature (°C) ---')
era5 = era5.dropna(subset=['temp_2m_c']).copy()
print(era5['temp_2m_c'].describe())

temp_bad = era5.loc[~era5['temp_2m_c'].between(5, 35), 'temp_2m_c']
assert len(temp_bad) == 0, f"Out-of-range ERA5 temp: {temp_bad.unique()[:5]}"


# --- ERA5 soil moisture layer 1 ---
print('\n--- ERA5 soil moisture layer 1 (m³/m³) ---')
era5 = era5.dropna(subset=['sm_era5_l1']).copy()
print(era5['sm_era5_l1'].describe())

sm_bad = era5.loc[~era5['sm_era5_l1'].between(0, 0.6), 'sm_era5_l1']
assert len(sm_bad) == 0, f"Out-of-range ERA5 soil moisture: {sm_bad.unique()[:5]}"


# --- FAPAR ---
print('\n--- FAPAR ---')
fapar = fapar.dropna(subset=['fapar']).copy()
print(fapar['fapar'].describe())

fapar_bad = fapar.loc[~fapar['fapar'].between(0, 1), 'fapar']
assert len(fapar_bad) == 0, f"Out-of-range FAPAR: {fapar_bad.unique()[:5]}"


# --- SMAP surface soil moisture ---
print('\n--- SMAP surface soil moisture (m³/m³) ---')
smap = smap.dropna(subset=['sm_surface']).copy()
print(smap['sm_surface'].describe())

smap_bad = smap.loc[~smap['sm_surface'].between(0, 0.6), 'sm_surface']
assert len(smap_bad) == 0, f"Out-of-range SMAP soil moisture: {smap_bad.unique()[:5]}"


print('\nAll range checks passed.')

--- MODIS NDVI and EVI ---
Dropping 4 rows with null MODIS values (known Oct 2009 / Oct 2015 sensor gaps)
Final MODIS rows: 9326
        ndvi_modis    evi_modis
count  9326.000000  9326.000000
mean      0.720662     0.457096
std       0.079547     0.072030
min       0.318030     0.174794
25%       0.688304     0.420220
50%       0.741729     0.472657
75%       0.774372     0.509444
max       0.876656     0.714828

--- Sentinel-2 NDVI ---
count    2160.000000
mean        0.643272
std         0.123664
min         0.026155
25%         0.605537
50%         0.679605
75%         0.724746
max         0.848162
Name: ndvi_s2, dtype: float64

--- CHIRPS precipitation (mm/month) ---
count    2160.000000
mean      171.580570
std        86.549967
min         1.361489
25%       107.831169
50%       156.751230
75%       233.680996
max       431.677210
Name: precip_mm, dtype: float64

--- ERA5 temperature (°C) ---
count    2160.000000
mean       17.978824
std         1.560495
min        13.430405
25% 

<b>Cross-file consistency checks</b>

In [ ]:
# 6a. CHIRPS vs ERA5 precipitation
m = chirps[['ward_name', 'month', 'precip_mm']].merge(
    era5[['ward_name', 'month', 'precip_era5_mm']],
    on=['ward_name', 'month'], how='inner'
)
r = m['precip_mm'].corr(m['precip_era5_mm'])
print(f"CHIRPS vs ERA5 rainfall: r = {r:.3f}")
assert r > 0.6, f"Rainfall datasets disagree (r={r})"

# 6b. S2 NDVI vs FAPAR
m = s2[['ward_name', 'month', 'ndvi_s2']].merge(
    fapar[['ward_name', 'month', 'fapar']],
    on=['ward_name', 'month'], how='inner'
)
r = m['ndvi_s2'].corr(m['fapar'])
print(f"S2 NDVI vs FAPAR: r = {r:.3f}")
assert r > 0.6, f"Vegetation datasets disagree (r={r})"

# 6c. S2 NDVI vs MODIS NDVI (where they overlap, 2020-2025)
modis_recent = modis[modis['month'] >= '2020-01-01']
m = s2[['ward_name', 'month', 'ndvi_s2']].merge(
    modis_recent[['ward_name', 'month', 'ndvi_modis']],
    on=['ward_name', 'month'], how='inner'
)
r = m['ndvi_s2'].corr(m['ndvi_modis'])
print(f"S2 NDVI vs MODIS NDVI (2020+): r = {r:.3f}")
assert r > 0.5, f"Cross-sensor NDVI disagree (r={r})"

# 6d. SMAP vs ERA5 surface soil moisture
m = smap[['ward_name', 'month', 'sm_surface']].merge(
    era5[['ward_name', 'month', 'sm_era5_l1']],
    on=['ward_name', 'month'], how='inner'
)
r = m['sm_surface'].corr(m['sm_era5_l1'])
print(f"SMAP vs ERA5 surface SM: r = {r:.3f}")

# Note: this one is expected to be weaker (different depths, resolutions).
# No hard assertion, but flag if r < 0.3.
if r < 0.3:
    print(f"  Warning: low agreement between soil moisture sources")

<b>Flag low-quality Sentinel-2 months</b>

In [ ]:
s2 = s2.sort_values(['ward_name', 'month']).reset_index(drop=True)
s2['typical_count'] = s2.groupby('ward_name')['ndvi_s2_count'].transform('median')
s2['quality_ratio'] = s2['ndvi_s2_count'] / s2['typical_count']
s2['low_quality'] = s2['quality_ratio'] < 0.3

n_bad = s2['low_quality'].sum()
print(f"Low-quality S2 months: {n_bad} / {len(s2)} ({100*n_bad/len(s2):.1f}%)")

# Per-ward summary
qc = (s2.groupby('ward_name')
        .agg(n_months=('month', 'count'),
             n_low=('low_quality', 'sum'),
             median_count=('ndvi_s2_count', 'median'),
             mean_ndvi=('ndvi_s2', 'mean'))
        .reset_index()
        .sort_values('n_low', ascending=False))
qc['pct_low'] = 100 * qc['n_low'] / qc['n_months']
print(qc.to_string(index=False))
qc.to_csv(RES / 'qc_summary.csv', index=False)

<b>Merge into one analysis frame</b>

In [ ]:
base = s2[['ward_name', 'ward_id', 'county', 'month',
           'ndvi_s2', 'ndvi_s2_std', 'ndvi_s2_count', 'low_quality']].copy()

for other, name in [(chirps, 'chirps'), (era5, 'era5'),
                    (fapar, 'fapar'), (smap, 'smap')]:
    value_cols = [c for c in other.columns
                  if c not in ('ward_name', 'ward_id', 'county', 'month')]
    base = base.merge(
        other[['ward_name', 'month'] + value_cols],
        on=['ward_name', 'month'],
        how='left',
        validate='one_to_one'
    )

base = base.merge(
    lc[['ward_name', 'lc_class']],
    on='ward_name', how='left', validate='many_to_one'
)

print('Merged shape:', base.shape)
print('Nulls per column:')
print(base.isnull().sum().sort_values(ascending=False).head(15))

<b>MODIS cleaned frame</b>

In [ ]:
modis_clean = modis[['ward_name', 'ward_id', 'month',
                     'ndvi_modis', 'ndvi_modis_std', 'ndvi_modis_count',
                     'evi_modis', 'evi_modis_std']].copy()
modis_clean = modis_clean.sort_values(['ward_name', 'month']).reset_index(drop=True)

print('MODIS frame:', modis_clean.shape)
print('Range:', modis_clean['month'].min().date(), 'to', modis_clean['month'].max().date())
print('Wards:', modis_clean['ward_name'].nunique())

<b>Save to Parquet</b>

In [ ]:
base.to_parquet(OUT / 'kericho_analysis_ready_2020_2025.parquet', index=False)
modis_clean.to_parquet(OUT / 'kericho_modis_2000_2025.parquet', index=False)

print('Saved:')
print('  kericho_analysis_ready_2020_2025.parquet', base.shape)
print('  kericho_modis_2000_2025.parquet', modis_clean.shape)

<b>Descriptive summary</b>

In [ ]:
cols = ['ndvi_s2', 'precip_mm', 'temp_2m_c', 'sm_era5_l1',
        'fapar', 'sm_surface']
summary = base[cols].describe().T
summary['missing_pct'] = 100 * (1 - summary['count'] / len(base))
print(summary.round(3))
summary.to_csv(RES / 'descriptive_summary.csv')